# 🧭 Customer Feedback Clustering & Product Roadmap Agent

**Problem:** Product managers are flooded with hundreds of feature requests from many channels, which makes patterns hard to see.

**Agent:** Claude (`claude-opus-5-5`) or Gemini reads every piece of feedback, **categorizes** it, **clusters** it by underlying need, and the code **counts and scores** each cluster (volume, distinct accounts, ARR, channel breadth, urgency) to surface the most-demanded updates as a Now / Next / Later roadmap.

**Architecture**
```
feedback CSV ─▶ Claude agent loop (tool runner) ──tools──▶ FeedbackStore
                 read_feedback · create_cluster · assign_feedback    (counts & scores
                 merge_clusters · get_unassigned · get_priorities     computed in code,
                 submit_roadmap                                        never by the LLM)
                                   │
                                   ▼
                     Streamlit dashboard  ·  Markdown / HTML / JSON report
```

### Before you start
1. Open the **🔑 Secrets** panel on the left of Colab and add a secret named `ANTHROPIC_API_KEY`, then turn on notebook access.
   *(No key? Everything still runs with the offline keyword baseline.)*
2. **Runtime → Run all**, then open the link printed by the last cell.

## 1 · Install dependencies

In [ ]:
!pip install -q 'anthropic>=1.13.0' 'google-genai>=1.30' 'streamlit>=1.50' pandas altair

## 2 · Write the project files
Each cell below writes one module. Edit them here and re-run to change the agent.

In [ ]:
!mkdir -p roadmap_agent data output .streamlit

In [ ]:
%%writefile roadmap_agent/__init__.py

**Feedback store** – loading, clusters, and the deterministic priority score.

In [ ]:
%%writefile roadmap_agent/store.py
"""Feedback store: loads raw feedback, holds clusters, and computes demand scores.

The LLM decides *what* belongs together (semantic judgment); this module does
all the *counting and scoring* deterministically, so numbers in the roadmap are
never hallucinated.
"""

from __future__ import annotations

import csv
import json
from dataclasses import dataclass, field
from datetime import date
from pathlib import Path

CATEGORIES = [
    "feature_request",
    "integration",
    "security_compliance",
    "performance",
    "ux_improvement",
    "bug",
    "not_a_request",  # praise, billing questions, etc. - excluded from the roadmap
]

# Categories that represent product work worth putting on a roadmap.
ROADMAP_CATEGORIES = {"feature_request", "integration", "security_compliance", "performance", "ux_improvement"}

# Weights for the priority score (sum to 1.0). Tune these to your strategy.
WEIGHTS = {
    "requests": 0.30,        # raw volume of mentions
    "accounts": 0.25,        # distinct customers asking
    "arr": 0.25,             # revenue of customers asking
    "channels": 0.10,        # breadth: asked for across many channels
    "urgency": 0.10,         # share of mentions flagged as blocking / churn risk
}


@dataclass
class Feedback:
    id: str
    date: str
    channel: str
    customer: str
    plan: str
    arr: float
    text: str


@dataclass
class Cluster:
    id: str
    name: str
    description: str
    category: str
    feedback_ids: list[str] = field(default_factory=list)
    urgent_ids: set[str] = field(default_factory=set)
    effort: str = "M"  # rough build effort: S, M or L


EFFORT_POINTS = {"S": 1, "M": 2, "L": 3}


class FeedbackStore:
    def __init__(self, feedback: list[Feedback]):
        self.feedback = {f.id: f for f in feedback}
        self.clusters: dict[str, Cluster] = {}
        self.assignment: dict[str, str] = {}  # feedback_id -> cluster_id
        self._next = 1

    # ---------- loading ----------
    @classmethod
    def from_file(cls, path: str | Path) -> "FeedbackStore":
        path = Path(path)
        if path.suffix == ".json":
            rows = json.loads(path.read_text())
        else:
            with path.open(newline="", encoding="utf-8") as fh:
                rows = list(csv.DictReader(fh))
        items = []
        for i, r in enumerate(rows, 1):
            items.append(
                Feedback(
                    id=str(r.get("id") or f"F{i:04d}"),
                    date=r.get("date", ""),
                    channel=r.get("channel", "unknown"),
                    customer=r.get("customer", "unknown"),
                    plan=r.get("plan", ""),
                    arr=float(r.get("arr") or 0),
                    text=r["text"].strip(),
                )
            )
        return cls(items)

    # ---------- cluster operations (exposed to the agent as tools) ----------
    def create_cluster(self, name: str, description: str, category: str) -> Cluster:
        if category not in CATEGORIES:
            raise ValueError(f"category must be one of {CATEGORIES}")
        for c in self.clusters.values():
            if c.name.lower() == name.lower():
                raise ValueError(f"cluster '{name}' already exists as {c.id}")
        cid = f"C{self._next:02d}"
        self._next += 1
        self.clusters[cid] = Cluster(cid, name, description, category)
        return self.clusters[cid]

    def assign(self, cluster_id: str, feedback_ids: list[str], urgent_ids: list[str] | None = None) -> dict:
        if cluster_id not in self.clusters:
            raise ValueError(f"unknown cluster {cluster_id}")
        unknown = [f for f in feedback_ids if f not in self.feedback]
        moved = []
        for fid in feedback_ids:
            if fid not in self.feedback:
                continue
            prev = self.assignment.get(fid)
            if prev and prev != cluster_id:
                self.clusters[prev].feedback_ids.remove(fid)
                self.clusters[prev].urgent_ids.discard(fid)
                moved.append(fid)
            if fid not in self.clusters[cluster_id].feedback_ids:
                self.clusters[cluster_id].feedback_ids.append(fid)
            self.assignment[fid] = cluster_id
        for fid in urgent_ids or []:
            if self.assignment.get(fid) == cluster_id:
                self.clusters[cluster_id].urgent_ids.add(fid)
        return {"cluster": cluster_id, "size": len(self.clusters[cluster_id].feedback_ids),
                "moved_from_other_clusters": moved, "unknown_ids": unknown}

    def merge(self, source_id: str, target_id: str) -> dict:
        src, tgt = self.clusters.get(source_id), self.clusters.get(target_id)
        if not src or not tgt or source_id == target_id:
            raise ValueError("need two distinct, existing cluster ids")
        urgent = list(src.urgent_ids)
        self.assign(target_id, list(src.feedback_ids), urgent)
        del self.clusters[source_id]
        return {"merged_into": target_id, "size": len(tgt.feedback_ids)}

    def set_effort(self, cluster_id: str, effort: str) -> None:
        if cluster_id in self.clusters and effort in EFFORT_POINTS:
            self.clusters[cluster_id].effort = effort

    def subset(self, keep: set[str]) -> "FeedbackStore":
        """A copy limited to the given feedback ids, with the same clusters (used for UI filters)."""
        sub = FeedbackStore([f for fid, f in self.feedback.items() if fid in keep])
        for c in self.clusters.values():
            ids = [f for f in c.feedback_ids if f in keep]
            sub.clusters[c.id] = Cluster(c.id, c.name, c.description, c.category, ids,
                                         {u for u in c.urgent_ids if u in keep}, c.effort)
            sub.assignment.update({f: c.id for f in ids})
        sub._next = self._next
        return sub

    def unassigned(self) -> list[str]:
        return [fid for fid in self.feedback if fid not in self.assignment]

    # ---------- analytics ----------
    def cluster_stats(self, c: Cluster) -> dict:
        items = [self.feedback[f] for f in c.feedback_ids]
        accounts = {i.customer for i in items}
        # ARR counted once per account, not once per mention.
        arr = sum({i.customer: i.arr for i in items}.values())
        dates = sorted(i.date for i in items if i.date)
        return {
            "id": c.id,
            "name": c.name,
            "category": c.category,
            "description": c.description,
            "requests": len(items),
            "accounts": len(accounts),
            "arr": arr,
            "channels": sorted({i.channel for i in items}),
            "urgent": len(c.urgent_ids),
            "effort": c.effort,
            "first_seen": dates[0] if dates else "",
            "last_seen": dates[-1] if dates else "",
            "quotes": [f"[{i.id} · {i.channel} · {i.customer}] {i.text}" for i in items[:3]],
        }

    def prioritize(self, weights: dict[str, float] | None = None) -> list[dict]:
        """Rank roadmap-worthy clusters with a transparent weighted score (0-100).

        `weights` overrides WEIGHTS (same keys); it is normalized so it need not sum to 1.
        """
        weights = weights or WEIGHTS
        total = sum(weights.values()) or 1
        weights = {k: v / total for k, v in weights.items()}
        stats = [self.cluster_stats(c) for c in self.clusters.values()
                 if c.category in ROADMAP_CATEGORIES and c.feedback_ids]
        if not stats:
            return []
        n_channels = len({f.channel for f in self.feedback.values()}) or 1
        mx = {k: max(s[k] for s in stats) or 1 for k in ("requests", "accounts", "arr")}
        for s in stats:
            parts = {
                "requests": s["requests"] / mx["requests"],
                "accounts": s["accounts"] / mx["accounts"],
                "arr": s["arr"] / mx["arr"],
                "channels": len(s["channels"]) / n_channels,
                "urgency": s["urgent"] / s["requests"],
            }
            s["score"] = round(100 * sum(weights.get(k, 0) * v for k, v in parts.items()), 1)
        stats.sort(key=lambda s: s["score"], reverse=True)
        for rank, s in enumerate(stats, 1):
            s["rank"] = rank
            s["tier"] = "Now" if s["score"] >= 60 else "Next" if s["score"] >= 35 else "Later"
        return stats

    def summary(self) -> dict:
        by_channel: dict[str, int] = {}
        for f in self.feedback.values():
            by_channel[f.channel] = by_channel.get(f.channel, 0) + 1
        return {
            "total_feedback": len(self.feedback),
            "assigned": len(self.assignment),
            "unassigned": len(self.unassigned()),
            "clusters": len(self.clusters),
            "by_channel": by_channel,
            "generated": date.today().isoformat(),
        }

**The Claude agent** – system prompt + tools; the SDK tool runner drives the loop.

In [ ]:
%%writefile roadmap_agent/agent.py
"""The clustering & roadmap agent, runnable on Claude or Gemini.

The model drives the loop: it reads feedback in batches, creates/merges clusters,
assigns every item, asks for the deterministic priority ranking, records a
rationale and effort estimate per theme, and finally submits a summary. The same
plain-Python tools serve both providers; only the loop driver differs.
"""

from __future__ import annotations

import functools
import json
import os

from .store import CATEGORIES, FeedbackStore

CLAUDE_MODEL = "claude-opus-5-5"
# Google's alias for the newest stable Flash model; override with the GEMINI_MODEL env var / secret.
GEMINI_MODEL = os.environ.get("GEMINI_MODEL", "gemini-flash-latest")
PROVIDERS = {"claude": "Claude", "gemini": "Gemini"}

SYSTEM_PROMPT = f"""You are a senior product-operations analyst. Product managers are flooded with
feature requests from many channels (app store reviews, support tickets, sales calls, NPS surveys,
social media, community forums). Your job is to turn that raw feedback into a clear, evidence-backed
product roadmap.

Work through it with your tools:
1. Call get_overview, then read_feedback in batches until you have seen every item.
2. Group items by the underlying user NEED, not surface wording ("night mode", "dark theme" and
   "white background is blinding" are one cluster; "SSO" and "SCIM provisioning" are related but
   distinct asks). Prefer specific, actionable clusters over vague buckets like "general requests".
   Categories: {", ".join(CATEGORIES)}. Use `bug` for defects and crashes, and `not_a_request` for
   praise, billing questions and similar items, so they stay out of the roadmap.
3. When assigning, pass urgent_ids for items that signal churn, a blocked deal or renewal, a
   compliance requirement, or a customer abandoning the product.
4. Merge clusters that turn out to be duplicates. Keep going until get_unassigned returns nothing.
5. Call get_priorities. The scores are computed in code from request volume, distinct accounts,
   revenue, channel breadth and urgency. Do not invent numbers; cite the ones the tool returns.
6. For every roadmap cluster, call set_theme_details with a 1-2 sentence rationale grounded in the
   numbers and quotes, and a rough build effort (S = days, M = a few weeks, L = a quarter or more)
   based on typical SaaS engineering cost. Note where the score alone is misleading (e.g. one huge
   account, or a cheap quick win).
7. Finish by calling submit_roadmap once with the executive summary."""


def build_tools(store: FeedbackStore, result: dict) -> list:
    """Plain functions with typed signatures and docstrings; each SDK derives its tool schema from them."""

    def get_overview() -> str:
        """Get totals: number of feedback items, how many are clustered, and counts per channel."""
        return json.dumps(store.summary())

    def read_feedback(offset: int = 0, limit: int = 40) -> str:
        """Read raw feedback items in a batch.

        Args:
            offset: Index of the first item to return.
            limit: Maximum number of items to return (max 60).
        """
        items = list(store.feedback.values())[offset: offset + min(limit, 60)]
        rows = [{"id": f.id, "channel": f.channel, "customer": f.customer, "plan": f.plan,
                 "text": f.text} for f in items]
        return json.dumps({"offset": offset, "returned": len(rows),
                           "total": len(store.feedback), "items": rows})

    def list_clusters() -> str:
        """List current clusters with their size and a few member ids."""
        return json.dumps([{"id": c.id, "name": c.name, "category": c.category,
                            "description": c.description, "size": len(c.feedback_ids),
                            "sample_ids": c.feedback_ids[:5]} for c in store.clusters.values()])

    def create_cluster(name: str, description: str, category: str) -> str:
        """Create a new cluster representing one underlying user need.

        Args:
            name: Short, specific name, e.g. "SAML SSO (Okta / Azure AD)".
            description: One sentence describing the need.
            category: One of feature_request, integration, security_compliance, performance,
                ux_improvement, bug, not_a_request.
        """
        try:
            c = store.create_cluster(name, description, category)
        except ValueError as e:
            return f"ERROR: {e}"
        return json.dumps({"id": c.id, "name": c.name})

    def assign_feedback(cluster_id: str, feedback_ids: list[str], urgent_ids: list[str] | None = None) -> str:
        """Assign feedback items to a cluster. Re-assigning an item moves it.

        Args:
            cluster_id: Target cluster id, e.g. "C03".
            feedback_ids: Feedback ids to put in this cluster.
            urgent_ids: Subset of feedback_ids signalling churn risk, a blocked deal, or a compliance need.
        """
        try:
            return json.dumps(store.assign(cluster_id, feedback_ids, urgent_ids))
        except ValueError as e:
            return f"ERROR: {e}"

    def merge_clusters(source_id: str, target_id: str) -> str:
        """Merge a duplicate cluster into another one; the source cluster is deleted.

        Args:
            source_id: Cluster to fold in and remove.
            target_id: Cluster that keeps all items.
        """
        try:
            return json.dumps(store.merge(source_id, target_id))
        except ValueError as e:
            return f"ERROR: {e}"

    def get_unassigned() -> str:
        """Return the full text of any feedback items not yet assigned to a cluster."""
        ids = store.unassigned()
        return json.dumps({"count": len(ids), "items": [
            {"id": i, "text": store.feedback[i].text} for i in ids]})

    def get_priorities() -> str:
        """Compute the ranked roadmap: score (0-100), tier (Now/Next/Later), requests, accounts, ARR, channels."""
        ranked = store.prioritize()
        return json.dumps([{k: s[k] for k in ("rank", "id", "name", "tier", "score", "requests",
                                                "accounts", "arr", "channels", "urgent")} for s in ranked])

    def set_theme_details(cluster_id: str, rationale: str, effort: str) -> str:
        """Record the rationale and estimated build effort for one roadmap cluster.

        Args:
            cluster_id: Cluster id, e.g. "C03".
            rationale: 1-2 sentences on why it ranks where it does, grounded in the numbers and quotes.
            effort: Estimated build effort: "S" (days), "M" (a few weeks) or "L" (a quarter or more).
        """
        if cluster_id not in store.clusters:
            return f"ERROR: unknown cluster {cluster_id}"
        result.setdefault("rationales", {})[cluster_id] = rationale
        store.set_effort(cluster_id, effort.strip().upper()[:1])
        return "Saved."

    def submit_roadmap(executive_summary: str) -> str:
        """Submit the final roadmap. Call exactly once, after set_theme_details for every roadmap cluster.

        Args:
            executive_summary: 3-5 sentences for a product leader: the top themes, what to build now, and key risks.
        """
        if store.unassigned():
            return f"ERROR: {len(store.unassigned())} items are still unassigned; call get_unassigned."
        result["executive_summary"] = executive_summary
        return "Roadmap saved. You are done."

    return [get_overview, read_feedback, list_clusters, create_cluster, assign_feedback,
            merge_clusters, get_unassigned, get_priorities, set_theme_details, submit_roadmap]


def _short(args: dict) -> str:
    s = json.dumps(args)
    return s[:100] + ("…" if len(s) > 100 else "")


def _run_claude(store: FeedbackStore, result: dict, on_event, api_key: str | None) -> None:
    import anthropic
    from anthropic import beta_tool

    client = anthropic.Anthropic(api_key=api_key) if api_key else anthropic.Anthropic()
    runner = client.beta.messages.tool_runner(
        model=CLAUDE_MODEL,
        max_tokens=16000,
        system=SYSTEM_PROMPT,
        thinking={"type": "adaptive"},
        output_config={"effort": "high"},
        # Server-side fallback if a request is ever declined by safety classifiers.
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
        max_iterations=80,
        tools=[beta_tool(fn) for fn in build_tools(store, result)],
        messages=[{"role": "user", "content":
                   "Cluster all of the customer feedback and produce a prioritized product roadmap."}],
    )
    for message in runner:
        if message.stop_reason == "refusal":
            raise RuntimeError(f"Request declined: {message.stop_details}")
        for block in message.content:
            if block.type == "tool_use":
                on_event(f"→ {block.name}({_short(block.input)})")
            elif block.type == "text" and block.text.strip():
                on_event(block.text.strip()[:300])


def _run_gemini(store: FeedbackStore, result: dict, on_event, api_key: str | None) -> None:
    from google import genai
    from google.genai import types

    def logged(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            on_event(f"→ {fn.__name__}({_short(kwargs)})")
            return fn(*args, **kwargs)
        return wrapper

    client = genai.Client(api_key=api_key) if api_key else genai.Client()
    # The SDK's automatic function calling runs the tool loop for us.
    chat = client.chats.create(
        model=GEMINI_MODEL,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            tools=[logged(fn) for fn in build_tools(store, result)],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=120),
        ),
    )
    response = chat.send_message("Cluster all of the customer feedback and produce a prioritized product roadmap.")
    # Gemini sometimes pauses to report progress before finishing; nudge it to complete the steps.
    for _ in range(3):
        if "executive_summary" in result:
            break
        if response.text:
            on_event(response.text.strip()[:300])
        response = chat.send_message("Continue with the remaining steps until you have called submit_roadmap.")


def run_agent(store: FeedbackStore, on_event=print, api_key: str | None = None, provider: str = "claude") -> dict:
    """Run the agent to completion. `on_event(str)` receives a log line per step (for CLI or UI)."""
    result: dict = {}
    if provider == "gemini":
        _run_gemini(store, result, on_event, api_key)
    else:
        _run_claude(store, result, on_event, api_key)
    if "executive_summary" not in result:
        raise RuntimeError("The agent stopped before submitting a roadmap. Try running it again.")
    return result

**Offline baseline** – keyword rules, no API key needed.

In [ ]:
%%writefile roadmap_agent/offline.py
"""Keyword-rule baseline: runs with no API key.

Useful for demos, tests, and as a comparison point that shows what the LLM
agent adds (it handles paraphrases and new themes these rules cannot).
"""

from __future__ import annotations

import re

from .store import FeedbackStore

# Rough build effort for the baseline (the Claude agent estimates its own).
EFFORTS = {
    "SCIM user provisioning": "M", "Single sign-on (SAML / Okta / Azure AD / Google)": "M",
    "Two-factor authentication": "S", "Audit logs": "M", "Dark mode": "S",
    "Slack & Microsoft Teams integration": "M", "Offline mode (mobile)": "L",
    "Public API, webhooks & rate limits": "L", "Custom dashboards & report builder": "L",
    "Data export (CSV / Excel / PDF)": "S", "Recurring tasks & templates": "S",
    "Localization": "M", "Speed & performance": "M",
}

# (name, category, description, keyword regex) - first match wins, so order matters.
RULES = [
    ("Crashes & freezes", "bug", "App crashes or freezes.", r"crash|freez"),
    ("Billing & account questions", "not_a_request", "Billing/account support, not product asks.", r"charged|refund|billing"),
    ("Praise", "not_a_request", "Positive feedback with no request.", r"^(great|love the|your onboarding)|keep it up"),
    ("SCIM user provisioning", "security_compliance", "Automatic user provisioning from the IdP.", r"scim|provisioning"),
    ("Single sign-on (SAML / Okta / Azure AD / Google)", "security_compliance", "Enterprise SSO login.", r"\bsso\b|single sign-on|saml|okta"),
    ("Two-factor authentication", "security_compliance", "MFA / 2FA, enforceable by admins.", r"2fa|mfa|two-factor"),
    ("Audit logs", "security_compliance", "Exportable log of admin actions.", r"audit log"),
    ("Dark mode", "ux_improvement", "Dark / night theme.", r"dark|night mode|white background"),
    ("Slack & Microsoft Teams integration", "integration", "Notifications and actions in chat tools.", r"slack|teams"),
    ("Offline mode (mobile)", "feature_request", "Use the app without connectivity.", r"offline|on flights|without signal|subway"),
    ("Public API, webhooks & rate limits", "integration", "Programmatic access and automation.", r"\bapi\b|webhook|rate limit"),
    ("Custom dashboards & report builder", "feature_request", "User-defined dashboards and reports.", r"dashboard|report|custom charts"),
    ("Data export (CSV / Excel / PDF)", "feature_request", "Export data in common formats.", r"export|csv|excel|pdf"),
    ("Recurring tasks & templates", "feature_request", "Repeat tasks and reusable templates.", r"recurring|repeat|template|reminder"),
    ("Localization", "feature_request", "UI in more languages.", r"spanish|portuguese|german|french|locali|translat"),
    ("Speed & performance", "performance", "Slow load times and sluggish UI.", r"slow|sluggish|forever|faster|performance|seconds"),
]

URGENT = r"block|deal|dealbreaker|renewal|risk|lost the|must-have|requires|required|contingent|auditor|compliance|deleting|ciso"


def cluster_offline(store: FeedbackStore) -> dict:
    ids: dict[str, str] = {}
    other = None
    for f in store.feedback.values():
        text = f.text.lower()
        for name, cat, desc, pattern in RULES:
            if re.search(pattern, text):
                if name not in ids:
                    ids[name] = store.create_cluster(name, desc, cat).id
                    store.set_effort(ids[name], EFFORTS.get(name, "M"))
                urgent = [f.id] if re.search(URGENT, text) else []
                store.assign(ids[name], [f.id], urgent)
                break
        else:
            other = other or store.create_cluster("Other / uncategorized", "No rule matched.", "feature_request").id
            store.assign(other, [f.id])
    ranked = store.prioritize()
    top = ", ".join(s["name"] for s in ranked[:3])
    return {
        "executive_summary": f"(Offline keyword baseline - run with an API key for the Claude agent's analysis.) "
                             f"Highest-demand themes: {top}.",
        "rationales": {},
    }

**Reports** – Markdown, HTML dashboard and JSON.

In [ ]:
%%writefile roadmap_agent/report.py
"""Render the roadmap as Markdown, JSON and a self-contained HTML dashboard."""

from __future__ import annotations

import json
from html import escape
from pathlib import Path

from .store import ROADMAP_CATEGORIES, FeedbackStore


def build(store: FeedbackStore, result: dict, weights: dict | None = None) -> dict:
    ranked = store.prioritize(weights)
    other = [store.cluster_stats(c) for c in store.clusters.values()
             if c.category not in ROADMAP_CATEGORIES and c.feedback_ids]
    for s in ranked:
        s["rationale"] = result.get("rationales", {}).get(s["id"], "")
    return {"summary": store.summary(), "executive_summary": result.get("executive_summary", ""),
            "roadmap": ranked, "excluded": other}


def write_all(report: dict, out_dir: str | Path) -> list[Path]:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    paths = [out / "roadmap.json", out / "roadmap.md", out / "roadmap.html"]
    paths[0].write_text(json.dumps(report, indent=2))
    paths[1].write_text(to_markdown(report))
    paths[2].write_text(to_html(report))
    return paths


def money(x: float) -> str:
    return f"${x / 1000:,.0f}k" if x >= 1000 else f"${x:,.0f}"


def to_markdown(r: dict) -> str:
    s = r["summary"]
    lines = [
        "# Product Roadmap from Customer Feedback", "",
        f"_{s['total_feedback']} feedback items · {len(s['by_channel'])} channels · "
        f"{s['clusters']} clusters · generated {s['generated']}_", "",
        "## Executive summary", "", r["executive_summary"], "",
        "## Prioritized roadmap", "",
        "| # | Tier | Theme | Score | Requests | Accounts | ARR asking | Channels | Urgent |",
        "|---|------|-------|------:|---------:|---------:|-----------:|---------:|-------:|",
    ]
    for x in r["roadmap"]:
        lines.append(f"| {x['rank']} | {x['tier']} | {x['name']} | {x['score']} | {x['requests']} | "
                     f"{x['accounts']} | {money(x['arr'])} | {len(x['channels'])} | {x['urgent']} |")
    lines += ["", "## Theme details", ""]
    for x in r["roadmap"]:
        lines += [f"### {x['rank']}. {x['name']} — {x['tier']} (score {x['score']})", "",
                  f"*{x['category']}* · {x['description']}", ""]
        if x.get("rationale"):
            lines += [f"**Why:** {x['rationale']}", ""]
        lines += [f"> {q}" for q in x["quotes"]] + [""]
    if r["excluded"]:
        lines += ["## Not on the roadmap (bugs & non-requests)", ""]
        lines += [f"- **{x['name']}** ({x['category']}): {x['requests']} items" for x in r["excluded"]]
    return "\n".join(lines) + "\n"


def to_html(r: dict) -> str:
    s = r["summary"]
    top = max((x["score"] for x in r["roadmap"]), default=1) or 1
    rows = []
    for x in r["roadmap"]:
        quotes = "".join(f"<li>{escape(q)}</li>" for q in x["quotes"])
        why = f"<p class='why'>{escape(x['rationale'])}</p>" if x.get("rationale") else ""
        rows.append(f"""
<details class="item">
  <summary>
    <span class="rank">{x['rank']}</span>
    <span class="tier t-{x['tier'].lower()}">{x['tier']}</span>
    <span class="name">{escape(x['name'])}</span>
    <span class="bar"><span style="width:{100 * x['score'] / top:.0f}%"></span></span>
    <span class="score">{x['score']}</span>
  </summary>
  <div class="body">
    <p class="meta">{x['requests']} requests · {x['accounts']} accounts · {money(x['arr'])} ARR asking ·
      {x['urgent']} urgent · {escape(', '.join(x['channels']))}</p>
    {why}
    <ul class="quotes">{quotes}</ul>
  </div>
</details>""")
    excluded = "".join(f"<li>{escape(x['name'])} <span class='muted'>({x['category']}, {x['requests']})</span></li>"
                       for x in r["excluded"])
    channels = " · ".join(f"{escape(k)} {v}" for k, v in sorted(s["by_channel"].items(), key=lambda kv: -kv[1]))
    return f"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Feedback Roadmap</title>
<style>
:root {{ --bg:#fafaf9; --card:#fff; --ink:#1c1917; --muted:#78716c; --line:#e7e5e4; --accent:#2563eb;
  --now:#dc2626; --next:#d97706; --later:#64748b; }}
@media (prefers-color-scheme: dark) {{ :root {{ --bg:#0c0a09; --card:#1c1917; --ink:#f5f5f4; --muted:#a8a29e;
  --line:#292524; --accent:#60a5fa; --now:#f87171; --next:#fbbf24; --later:#94a3b8; }} }}
* {{ box-sizing:border-box }}
body {{ margin:0; background:var(--bg); color:var(--ink); font:15px/1.5 system-ui,-apple-system,sans-serif }}
main {{ max-width:900px; margin:0 auto; padding:32px 16px }}
h1 {{ margin:0 0 4px; font-size:26px }} h2 {{ font-size:17px; margin:32px 0 12px }}
.muted, .meta {{ color:var(--muted) }}
.kpis {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(140px,1fr)); gap:12px; margin:20px 0 }}
.kpi, .summary, .item {{ background:var(--card); border:1px solid var(--line); border-radius:10px }}
.kpi {{ padding:14px }} .kpi b {{ display:block; font-size:24px }}
.summary {{ padding:16px 18px }}
.item {{ margin-bottom:8px }}
.item summary {{ display:grid; grid-template-columns:28px 56px 1fr 140px 44px; gap:10px; align-items:center;
  padding:12px 14px; cursor:pointer; list-style:none }}
.rank {{ color:var(--muted); font-variant-numeric:tabular-nums }}
.tier {{ font-size:12px; font-weight:600; text-align:center; border-radius:99px; padding:2px 0; color:#fff }}
.t-now {{ background:var(--now) }} .t-next {{ background:var(--next) }} .t-later {{ background:var(--later) }}
.name {{ font-weight:600 }}
.bar {{ height:8px; background:var(--line); border-radius:99px; overflow:hidden }}
.bar span {{ display:block; height:100%; background:var(--accent) }}
.score {{ text-align:right; font-variant-numeric:tabular-nums; font-weight:600 }}
.body {{ padding:0 14px 14px 52px }} .why {{ margin:6px 0 }}
.quotes {{ margin:6px 0 0; padding-left:18px; color:var(--muted); font-size:14px }}
@media (max-width:600px) {{ .item summary {{ grid-template-columns:24px 52px 1fr 40px }} .bar {{ display:none }}
  .body {{ padding-left:14px }} }}
</style></head><body><main>
<h1>Product roadmap from customer feedback</h1>
<div class="muted">Generated {s['generated']} · {channels}</div>
<div class="kpis">
  <div class="kpi"><b>{s['total_feedback']}</b><span class="muted">feedback items</span></div>
  <div class="kpi"><b>{len(s['by_channel'])}</b><span class="muted">channels</span></div>
  <div class="kpi"><b>{s['clusters']}</b><span class="muted">clusters</span></div>
  <div class="kpi"><b>{sum(1 for x in r['roadmap'] if x['tier'] == 'Now')}</b><span class="muted">build-now themes</span></div>
</div>
<h2>Executive summary</h2>
<div class="summary">{escape(r['executive_summary'])}</div>
<h2>Prioritized themes <span class="muted">(click to see evidence)</span></h2>
{''.join(rows)}
<h2>Not on the roadmap</h2>
<ul>{excluded}</ul>
<p class="muted">Score = 30% request volume + 25% distinct accounts + 25% ARR + 10% channel breadth + 10% urgency.
Now ≥ 60 · Next ≥ 35 · Later &lt; 35.</p>
</main></body></html>
"""

**Ask the agent & PRD drafts** – streams follow-up answers and PRDs from Claude.

In [ ]:
%%writefile roadmap_agent/assistant.py
"""Follow-up Q&A and PRD drafting over a finished roadmap (streams text for the UI)."""

from __future__ import annotations

import json
from collections.abc import Iterator

from .agent import CLAUDE_MODEL, GEMINI_MODEL
from .store import FeedbackStore

ASK_SYSTEM = """You are a product-operations analyst answering a product manager's questions about a
customer-feedback analysis. You get the ranked roadmap (scores are computed in code from request
volume, distinct accounts, ARR, channel breadth and urgency) and every raw feedback item with its
cluster. Answer from this data only: cite numbers and feedback ids (e.g. F014), say when the data
can't answer something, and keep answers short and scannable."""

PRD_SYSTEM = """You are a senior product manager. Write a concise one-page PRD in Markdown for the
theme you are given, grounded only in the customer evidence provided. Sections: Problem, Who is
asking (with the numbers), Goals & non-goals, User stories (3-6, "As a ... I want ... so that ..."),
Requirements (must / should), Success metrics, Open questions. Quote customers with their feedback
ids. Do not invent customers, numbers or commitments."""


def _context(store: FeedbackStore, ranked: list[dict]) -> str:
    roadmap = [{k: s[k] for k in ("rank", "id", "name", "tier", "score", "requests", "accounts",
                                  "arr", "channels", "urgent", "effort", "category")} for s in ranked]
    items = [{"id": f.id, "date": f.date, "channel": f.channel, "customer": f.customer, "plan": f.plan,
              "arr": f.arr, "cluster": store.clusters[store.assignment[f.id]].name
              if f.id in store.assignment else None, "text": f.text} for f in store.feedback.values()]
    return f"<roadmap>\n{json.dumps(roadmap)}\n</roadmap>\n<feedback>\n{json.dumps(items)}\n</feedback>"


def _stream(provider: str, api_key: str, system: str, messages: list[dict]) -> Iterator[str]:
    """Stream text from Claude or Gemini. `messages` is [{"role": "user"|"assistant", "content": str}]."""
    if provider == "gemini":
        yield from _stream_gemini(api_key, system, messages)
    else:
        yield from _stream_claude(api_key, system, messages)


def _stream_claude(api_key: str, system: str, messages: list[dict]) -> Iterator[str]:
    import anthropic

    client = anthropic.Anthropic(api_key=api_key)
    with client.beta.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=8000,
        system=system,
        thinking={"type": "adaptive"},
        output_config={"effort": "medium"},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
        messages=messages,
    ) as stream:
        yield from stream.text_stream
        final = stream.get_final_message()
    if final.stop_reason == "refusal":
        yield "\n\n_The model declined to answer this request._"
    elif final.stop_reason == "max_tokens":
        yield "\n\n_(Answer cut off at the length limit.)_"


def _stream_gemini(api_key: str, system: str, messages: list[dict]) -> Iterator[str]:
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=api_key)
    contents = [types.Content(role="model" if m["role"] == "assistant" else "user",
                              parts=[types.Part(text=m["content"])]) for m in messages]
    for chunk in client.models.generate_content_stream(
            model=GEMINI_MODEL, contents=contents,
            config=types.GenerateContentConfig(system_instruction=system)):
        if chunk.text:
            yield chunk.text


def ask(provider: str, api_key: str, store: FeedbackStore, ranked: list[dict], history: list[dict]) -> Iterator[str]:
    """Stream an answer. `history` is the chat so far as [{"role", "content"}], ending with the user turn."""
    # The data goes in the first user turn so it is identical (and cacheable) across follow-ups.
    first = {"role": "user", "content": _context(store, ranked) + "\n\n" + history[0]["content"]}
    return _stream(provider, api_key, ASK_SYSTEM, [first, *history[1:]])


def draft_prd(provider: str, api_key: str, store: FeedbackStore, theme: dict) -> Iterator[str]:
    cluster = store.clusters[theme["id"]]
    evidence = [{"id": f.id, "channel": f.channel, "customer": f.customer, "plan": f.plan,
                 "urgent": f.id in cluster.urgent_ids, "text": f.text}
                for f in (store.feedback[i] for i in cluster.feedback_ids)]
    stats = {k: theme[k] for k in ("name", "description", "tier", "score", "requests", "accounts",
                                   "arr", "channels", "urgent", "effort")}
    prompt = f"Theme:\n{json.dumps(stats)}\n\nCustomer evidence:\n{json.dumps(evidence)}\n\nWrite the PRD."
    return _stream(provider, api_key, PRD_SYSTEM, [{"role": "user", "content": prompt}])

**CLI** entry point.

In [ ]:
%%writefile main.py
"""CLI: python main.py data/feedback.csv [--offline] [--out output]"""

import argparse

from roadmap_agent import report
from roadmap_agent.store import FeedbackStore


def main():
    p = argparse.ArgumentParser(description="Cluster customer feedback and build a prioritized roadmap.")
    p.add_argument("input", help="CSV or JSON file with a 'text' column (optional: id, date, channel, customer, plan, arr)")
    p.add_argument("--out", default="output", help="output directory")
    p.add_argument("--offline", action="store_true", help="use the keyword baseline instead of an AI model")
    p.add_argument("--provider", choices=["claude", "gemini"], default="claude",
                   help="AI provider (needs ANTHROPIC_API_KEY or GEMINI_API_KEY)")
    args = p.parse_args()

    store = FeedbackStore.from_file(args.input)
    print(f"Loaded {len(store.feedback)} feedback items from {args.input}")

    if args.offline:
        from roadmap_agent.offline import cluster_offline
        result = cluster_offline(store)
    else:
        from roadmap_agent.agent import run_agent
        print(f"Running the {args.provider} agent…")
        result = run_agent(store, provider=args.provider)

    rep = report.build(store, result)
    for path in report.write_all(rep, args.out):
        print(f"Wrote {path}")
    print("\nTop themes:")
    for x in rep["roadmap"][:5]:
        print(f"  {x['rank']}. [{x['tier']:5}] {x['name']}  (score {x['score']}, "
              f"{x['requests']} requests, {x['accounts']} accounts)")


if __name__ == "__main__":
    main()

**Theme** – light and dark variants of the app's look.

In [ ]:
%%writefile .streamlit/config.toml
# Modern SaaS theme. Light and dark variants follow the viewer's system setting.
[theme]
baseRadius = "0.6rem"
showWidgetBorder = true
chartCategoricalColors = ["#4F46E5", "#0EA5E9", "#94A3B8", "#14B8A6", "#F59E0B", "#EC4899"]

[theme.light]
primaryColor = "#4F46E5"
backgroundColor = "#F8FAFC"
secondaryBackgroundColor = "#FFFFFF"
textColor = "#0F172A"
borderColor = "#E2E8F0"
linkColor = "#4F46E5"

[theme.dark]
primaryColor = "#818CF8"
backgroundColor = "#0B0D12"
secondaryBackgroundColor = "#141821"
textColor = "#E5E7EB"
borderColor = "#262B36"
linkColor = "#A5B4FC"

[theme.light.sidebar]
backgroundColor = "#FFFFFF"

[theme.dark.sidebar]
backgroundColor = "#10131A"

[client]
toolbarMode = "minimal"

[browser]
gatherUsageStats = false

**Streamlit front-end (Signal)** – overview, roadmap board, priority matrix, insights, evidence + PRDs, chat, export.

In [ ]:
%%writefile app.py
"""Signal: Streamlit front-end for the Customer Feedback Clustering & Roadmap Agent.

Run:  streamlit run app.py
"""

import json
import os
import tempfile
from html import escape
from pathlib import Path

import altair as alt
import pandas as pd
import streamlit as st

from roadmap_agent import report
from roadmap_agent.agent import PROVIDERS
from roadmap_agent.offline import cluster_offline
from roadmap_agent.store import EFFORT_POINTS, WEIGHTS, FeedbackStore

SAMPLE = Path(__file__).parent / "data" / "feedback.csv"
TIERS = {"Now": "#4F46E5", "Next": "#0EA5E9", "Later": "#94A3B8"}
EFFORT_LABEL = {"S": "Small · days", "M": "Medium · weeks", "L": "Large · quarter+"}
WEIGHT_LABELS = {"requests": "Request volume", "accounts": "Distinct accounts", "arr": "Revenue (ARR)",
                 "channels": "Channel breadth", "urgency": "Urgency"}
SUGGESTIONS = ["Why is the #1 theme ranked first?", "What do enterprise customers want most?",
               "Which themes are quick wins?", "Which requests signal churn risk?"]

st.set_page_config(page_title="Signal · Feedback → Roadmap", page_icon="🧭", layout="wide",
                   initial_sidebar_state="expanded")

# ---------------------------------------------------------------- styling
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');
html, body, [data-testid="stAppViewContainer"], [data-testid="stSidebar"], button, input, textarea {
  font-family: 'Inter', system-ui, -apple-system, sans-serif !important; }
[data-testid="stAppViewContainer"] .block-container { padding-top: 2rem; max-width: 1280px; }
h1, h2, h3 { letter-spacing: -0.02em; }
.hero { padding: 28px 32px; border-radius: 16px; margin-bottom: 20px;
  background: linear-gradient(135deg, rgba(79,70,229,.12), rgba(14,165,233,.08));
  border: 1px solid rgba(99,102,241,.25); }
.hero h1 { font-size: 2.1rem; font-weight: 800; margin: 0 0 6px; padding: 0; }
.hero p { margin: 0; opacity: .75; font-size: 1.02rem; max-width: 760px; }
.pills { margin-top: 14px; display: flex; gap: 8px; flex-wrap: wrap; }
.pill { font-size: 12px; font-weight: 600; padding: 4px 10px; border-radius: 999px;
  background: rgba(127,127,127,.12); }
.pill.live { background: rgba(79,70,229,.15); color: #6366F1; }
.card { padding: 16px 18px; border-radius: 12px; border: 1px solid rgba(127,127,127,.2);
  background: rgba(127,127,127,.04); height: 100%; }
.kpi .label { font-size: 12px; font-weight: 600; text-transform: uppercase; letter-spacing: .05em; opacity: .6; }
.kpi .value { font-size: 1.9rem; font-weight: 800; margin-top: 2px; }
.kpi .sub { font-size: 12.5px; opacity: .6; }
.step .num { width: 28px; height: 28px; border-radius: 8px; display: inline-flex; align-items: center;
  justify-content: center; background: rgba(79,70,229,.15); color: #6366F1; font-weight: 700; margin-bottom: 8px; }
.step h4 { margin: 0 0 4px; font-size: 1rem; } .step p { margin: 0; opacity: .7; font-size: .92rem; }
.theme { padding: 14px 16px; border-radius: 12px; border: 1px solid rgba(127,127,127,.2);
  background: rgba(127,127,127,.04); margin-bottom: 10px; border-left: 4px solid var(--tier); }
.theme .top { display: flex; justify-content: space-between; align-items: baseline; gap: 8px; }
.theme .name { font-weight: 700; font-size: .98rem; line-height: 1.3; }
.theme .score { font-weight: 800; color: var(--tier); font-size: 1.1rem; }
.theme .meta { font-size: 12.5px; opacity: .65; margin-top: 6px; }
.chip { display: inline-block; font-size: 11px; font-weight: 600; padding: 2px 8px; border-radius: 6px;
  background: rgba(127,127,127,.13); margin: 6px 4px 0 0; }
.col-head { display: flex; align-items: center; gap: 8px; font-weight: 700; margin-bottom: 10px; }
.dot { width: 10px; height: 10px; border-radius: 50%; background: var(--tier); }
.summary { padding: 18px 20px; border-radius: 12px; border: 1px solid rgba(99,102,241,.3);
  background: rgba(79,70,229,.06); line-height: 1.6; }
.quote { padding: 10px 14px; border-left: 3px solid rgba(127,127,127,.35); margin: 6px 0;
  font-size: .93rem; opacity: .85; }
.quote b { font-weight: 600; opacity: .7; font-size: 12px; }
div[data-testid="stTabs"] button p { font-weight: 600; }
</style>
""", unsafe_allow_html=True)

ss = st.session_state


def html(s: str) -> None:
    st.markdown(s, unsafe_allow_html=True)


def money(x: float) -> str:
    return f"${x / 1e6:,.1f}M" if x >= 1e6 else f"${x / 1000:,.0f}k" if x >= 1000 else f"${x:,.0f}"


def nice(s: str) -> str:
    return s.replace("_", " ").title()


def kpi(col, label: str, value, sub: str = "") -> None:
    col.markdown(f"<div class='card kpi'><div class='label'>{label}</div><div class='value'>{value}</div>"
                 f"<div class='sub'>{sub}</div></div>", unsafe_allow_html=True)


def theme_card(x: dict, show_rank: bool = True) -> None:
    rank = f"#{x['rank']} · " if show_rank else ""
    chips = f"<span class='chip'>{EFFORT_LABEL.get(x['effort'], x['effort'])}</span>"
    if x["urgent"]:
        chips += f"<span class='chip'>⚡ {x['urgent']} urgent</span>"
    html(f"""<div class='theme' style='--tier:{TIERS[x["tier"]]}'>
      <div class='top'><span class='name'>{rank}{escape(x['name'])}</span><span class='score'>{x['score']:.0f}</span></div>
      <div class='meta'>{x['requests']} requests · {x['accounts']} accounts · {money(x['arr'])} ARR · {len(x['channels'])} channels</div>
      {chips}</div>""")


# ---------------------------------------------------------------- sidebar
with st.sidebar:
    html("<div style='font-size:1.35rem;font-weight:800;letter-spacing:-.02em'>🧭 Signal</div>"
         "<div style='opacity:.6;font-size:.85rem;margin-bottom:8px'>Feedback → Roadmap agent</div>")

    st.subheader("Data", divider="gray")
    upload = st.file_uploader("Upload feedback (CSV or JSON)", type=["csv", "json"],
                              help="Needs a `text` column. Optional: id, date, channel, customer, plan, arr.")
    st.caption("No file? The sample dataset is used (80 requests from 6 channels).")
    st.download_button("Download CSV template", SAMPLE.read_bytes(), "feedback_template.csv", "text/csv",
                       width="stretch")

    st.subheader("AI provider", divider="gray")
    provider_name = st.segmented_control("AI provider", list(PROVIDERS.values()), default="Claude",
                                         label_visibility="collapsed",
                                         help="Used by the AI agent, Ask the agent, and PRD drafts.")
    provider = "gemini" if provider_name == "Gemini" else "claude"
    if provider == "gemini":
        server_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY", "")
        key_label, key_hint = "Gemini API key", "AIza…"
        st.caption("Get a free key at [aistudio.google.com](https://aistudio.google.com/app/apikey).")
    else:
        server_key = os.environ.get("ANTHROPIC_API_KEY", "")
        key_label, key_hint = "Anthropic API key", "sk-ant-…"
    user_key = st.text_input(key_label, type="password", key=f"key_{provider}",
                             placeholder="Using the server's key" if server_key else key_hint,
                             help="A key set on the server is never sent to your browser.")
    api_key = user_key or server_key
    ai = PROVIDERS[provider]

    st.subheader("Clustering engine", divider="gray")
    engine = st.segmented_control("Clustering engine", ["AI agent", "Offline baseline"],
                                  default="AI agent", label_visibility="collapsed",
                                  help="The offline baseline uses keyword rules and needs no API key.")
    run = st.button("Build roadmap", type="primary", width="stretch", icon="✨")


def load_store() -> FeedbackStore:
    if upload is None:
        return FeedbackStore.from_file(SAMPLE)
    with tempfile.NamedTemporaryFile(delete=False, suffix=Path(upload.name).suffix) as tmp:
        tmp.write(upload.getvalue())
    return FeedbackStore.from_file(tmp.name)


try:
    raw_store = load_store()
except Exception as e:
    st.error(f"Couldn't read that file: {e}")
    st.stop()

# ---------------------------------------------------------------- run the agent
if run:
    if engine == "AI agent" and not api_key:
        st.sidebar.error("Add an API key, or switch to the offline baseline.")
    else:
        try:
            if engine == "AI agent":
                from roadmap_agent.agent import run_agent
                with st.status(f"{ai} is reading and clustering your feedback…", expanded=True) as status:
                    result = run_agent(raw_store, on_event=lambda line: st.caption(line), api_key=api_key,
                                       provider=provider)
                    status.update(label="Roadmap ready", state="complete", expanded=False)
            else:
                result = cluster_offline(raw_store)
            ss.update(store=raw_store, result=result, chat=[], prds={},
                      engine=f"{ai} agent" if engine == "AI agent" else "Offline baseline")
        except Exception as e:
            st.error(f"Run failed: {e}")

# ---------------------------------------------------------------- landing page
if "store" not in ss:
    html("""<div class='hero'><h1>Turn feedback noise into a roadmap.</h1>
      <p>Signal reads every feature request from app reviews, support tickets, sales calls, surveys and social,
      clusters them by the underlying need, and ranks what customers want most, backed by the evidence.</p>
      <div class='pills'><span class='pill'>Claude or Gemini clustering</span><span class='pill'>Transparent scoring</span>
      <span class='pill'>Now / Next / Later roadmap</span><span class='pill'>PRD drafts</span></div></div>""")
    steps = [("Ingest", "Upload a CSV/JSON export from any channel, or use the sample data."),
             ("Cluster", "The agent groups requests by user need and flags churn and deal risks."),
             ("Prioritize", "Code scores each theme on volume, accounts, revenue, breadth and urgency."),
             ("Act", "Explore the board, the matrix and trends; ask follow-ups; draft PRDs.")]
    for col, (i, (title, text)) in zip(st.columns(4), enumerate(steps, 1)):
        col.markdown(f"<div class='card step'><div class='num'>{i}</div><h4>{title}</h4><p>{text}</p></div>",
                     unsafe_allow_html=True)
    st.write("")
    st.info("Choose an engine in the sidebar and click **Build roadmap**.", icon="👈")
    st.markdown(f"##### Preview · {len(raw_store.feedback)} feedback items")
    st.dataframe(pd.DataFrame([vars(f) for f in raw_store.feedback.values()]), width="stretch",
                 hide_index=True, height=320,
                 column_config={"arr": st.column_config.NumberColumn("ARR", format="$%d"),
                                "text": st.column_config.TextColumn("Feedback", width="large")})
    st.stop()

# ---------------------------------------------------------------- weights & filters (after a run)
store: FeedbackStore = ss.store
all_fb = list(store.feedback.values())


def reset_weights() -> None:
    for k, v in WEIGHTS.items():
        ss[f"w_{k}"] = int(v * 100)


with st.sidebar:
    st.subheader("Scoring weights", divider="gray")
    st.caption("Drag to re-rank the roadmap live.")
    weights = {k: st.slider(WEIGHT_LABELS[k], 0, 100, int(v * 100), 5, key=f"w_{k}") for k, v in WEIGHTS.items()}
    st.button("Reset weights", on_click=reset_weights, width="stretch")

    st.subheader("Filters", divider="gray")
    channels = sorted({f.channel for f in all_fb})
    plans = sorted({f.plan for f in all_fb if f.plan})
    f_channels = st.multiselect("Channels", channels, default=channels, format_func=nice)
    f_plans = st.multiselect("Plans", plans, default=plans, format_func=nice)
    dates = sorted(f.date for f in all_fb if f.date)
    f_dates = None
    if dates:
        d0, d1 = pd.to_datetime(dates[0]).date(), pd.to_datetime(dates[-1]).date()
        if d0 < d1:
            f_dates = st.slider("Date range", d0, d1, (d0, d1))

keep = {f.id for f in all_fb
        if f.channel in f_channels and (not plans or f.plan in f_plans)
        and (not f_dates or not f.date or f_dates[0] <= pd.to_datetime(f.date).date() <= f_dates[1])}
view = store.subset(keep)
rep = report.build(view, ss.result, weights)
ranked, summary = rep["roadmap"], rep["summary"]
filtered = len(keep) < len(all_fb)

# ---------------------------------------------------------------- header
engine_pill = ss.engine
html(f"""<div class='hero'><h1>Product roadmap</h1>
  <p>{summary['total_feedback']} feedback items from {len(summary['by_channel'])} channels, grouped into
  {len(ranked)} roadmap themes.</p>
  <div class='pills'><span class='pill live'>● {engine_pill}</span><span class='pill'>Generated {summary['generated']}</span>
  {"<span class='pill'>Filtered view</span>" if filtered else ""}
  {"<span class='pill'>Custom weights</span>" if any(weights[k] != int(WEIGHTS[k] * 100) for k in WEIGHTS) else ""}</div></div>""")

if not ranked:
    st.warning("No roadmap themes match the current filters.")
    st.stop()

tabs = st.tabs(["Overview", "Roadmap board", "Priority matrix", "Insights", "Evidence", "Ask the agent", "Export"])

# ---------------------------------------------------------------- overview
with tabs[0]:
    now = [x for x in ranked if x["tier"] == "Now"]
    urgent = sum(x["urgent"] for x in ranked)
    asking_arr = sum({f.customer: f.arr for f in view.feedback.values()
                      if f.id in view.assignment and view.clusters[view.assignment[f.id]].category
                      in report.ROADMAP_CATEGORIES}.values())
    c = st.columns(5)
    kpi(c[0], "Feedback analysed", summary["total_feedback"], f"{len(summary['by_channel'])} channels")
    kpi(c[1], "Roadmap themes", len(ranked), f"{len(rep['excluded'])} non-roadmap groups")
    kpi(c[2], "Build now", len(now), "score ≥ 60")
    kpi(c[3], "Urgent signals", urgent, "churn, blocked deals, compliance")
    kpi(c[4], "ARR asking", money(asking_arr), "distinct accounts")
    st.write("")

    left, right = st.columns([3, 2], gap="large")
    with left:
        st.markdown("##### Executive summary")
        html(f"<div class='summary'>{escape(rep['executive_summary'] or '—')}</div>")
        st.write("")
        st.markdown("##### Ranking")
        df = pd.DataFrame([{"Theme": x["name"], "Score": x["score"], "Tier": x["tier"], "Requests": x["requests"],
                            "Accounts": x["accounts"], "ARR": x["arr"]} for x in ranked])
        bars = alt.Chart(df).mark_bar(cornerRadiusEnd=4, height=18).encode(
            x=alt.X("Score:Q", scale=alt.Scale(domain=[0, 100]), title=None),
            y=alt.Y("Theme:N", sort=list(df["Theme"]), title=None, axis=alt.Axis(labelLimit=300)),
            color=alt.Color("Tier:N", scale=alt.Scale(domain=list(TIERS), range=list(TIERS.values())),
                            legend=alt.Legend(orient="top", title=None)),
            tooltip=["Theme", "Score", "Tier", "Requests", "Accounts", alt.Tooltip("ARR:Q", format="$,.0f")])
        labels = bars.mark_text(align="left", dx=4, fontWeight=600).encode(text=alt.Text("Score:Q", format=".0f"),
                                                                             color=alt.value("#94A3B8"))
        st.altair_chart((bars + labels).properties(height=30 * len(df) + 30), width="stretch")
    with right:
        st.markdown("##### Top priorities")
        for x in ranked[:4]:
            theme_card(x)
            if x.get("rationale"):
                st.caption(x["rationale"])

# ---------------------------------------------------------------- roadmap board
with tabs[1]:
    st.caption("Themes by tier. Tiers come from the score: Now ≥ 60, Next ≥ 35, Later < 35.")
    for col, tier in zip(st.columns(3, gap="medium"), TIERS):
        items = [x for x in ranked if x["tier"] == tier]
        with col:
            html(f"<div class='col-head' style='--tier:{TIERS[tier]}'><span class='dot'></span>{tier}"
                 f"<span class='pill'>{len(items)}</span></div>")
            for x in items:
                theme_card(x)
            if not items:
                st.caption("Nothing here with the current weights and filters.")

# ---------------------------------------------------------------- priority matrix
with tabs[2]:
    st.caption("Impact (score) against estimated build effort. Bubble size = ARR asking.")
    mdf = pd.DataFrame([{"Theme": x["name"], "Score": x["score"], "Tier": x["tier"], "Effort": x["effort"],
                         "x": EFFORT_POINTS.get(x["effort"], 2) + ((x["rank"] % 5) - 2) * 0.09,
                         "ARR": max(x["arr"], 1), "Requests": x["requests"]} for x in ranked])
    quad = pd.DataFrame([{"x": 1.0, "y": 97, "t": "QUICK WINS"}, {"x": 3.0, "y": 97, "t": "BIG BETS"},
                         {"x": 1.0, "y": 3, "t": "FILL-INS"}, {"x": 3.0, "y": 3, "t": "MONEY PITS"}])
    base_x = alt.X("x:Q", scale=alt.Scale(domain=[0.5, 4.4]), title="Estimated effort",
                   axis=alt.Axis(values=[1, 2, 3], grid=False, labelExpr="datum.value == 1 ? 'Small' : datum.value == 2 ? 'Medium' : 'Large'"))
    points = alt.Chart(mdf).mark_circle(opacity=.85, stroke="white", strokeWidth=1).encode(
        x=base_x, y=alt.Y("Score:Q", scale=alt.Scale(domain=[0, 100]), title="Impact score", axis=alt.Axis(grid=False)),
        size=alt.Size("ARR:Q", scale=alt.Scale(range=[120, 1800]), legend=None),
        color=alt.Color("Tier:N", scale=alt.Scale(domain=list(TIERS), range=list(TIERS.values())),
                        legend=alt.Legend(orient="top", title=None)),
        tooltip=["Theme", "Score", "Tier", "Effort", "Requests", alt.Tooltip("ARR:Q", format="$,.0f")])
    text = alt.Chart(mdf).mark_text(align="left", dx=12, fontSize=11).encode(x=base_x, y="Score:Q", text="Theme:N")
    rules = (alt.Chart(pd.DataFrame({"y": [50]})).mark_rule(strokeDash=[4, 4], opacity=.4).encode(y="y:Q") +
             alt.Chart(pd.DataFrame({"x": [2.0]})).mark_rule(strokeDash=[4, 4], opacity=.4).encode(x="x:Q"))
    qlabels = alt.Chart(quad).mark_text(fontWeight=700, fontSize=11, opacity=.4).encode(x="x:Q", y="y:Q", text="t:N")
    st.altair_chart((rules + qlabels + points + text).properties(height=520), width="stretch")

    wins = [x for x in ranked if x["effort"] == "S" and x["score"] >= 35]
    bets = [x for x in ranked if x["effort"] == "L" and x["score"] >= 50]
    a, b = st.columns(2, gap="large")
    with a:
        st.markdown("##### ⚡ Quick wins")
        st.caption("High enough demand, small effort: ship these early.")
        for x in wins or []:
            theme_card(x)
        if not wins:
            st.caption("None right now.")
    with b:
        st.markdown("##### 🎯 Big bets")
        st.caption("High demand, large effort: plan and staff deliberately.")
        for x in bets:
            theme_card(x)
        if not bets:
            st.caption("None right now.")

# ---------------------------------------------------------------- insights
with tabs[3]:
    rows = [{"id": f.id, "date": f.date, "channel": nice(f.channel), "customer": f.customer, "plan": nice(f.plan or "unknown"),
             "arr": f.arr, "theme": view.clusters[view.assignment[f.id]].name}
            for f in view.feedback.values() if f.id in view.assignment]
    idf = pd.DataFrame(rows)
    names = [x["name"] for x in ranked]
    idf = idf[idf["theme"].isin(names)]
    top = names[:6]

    a, b = st.columns(2, gap="large")
    with a:
        st.markdown("##### Momentum")
        st.caption("Cumulative requests over time for the top 6 themes. Steeper = growing demand.")
        tdf = idf[idf["theme"].isin(top) & (idf["date"] != "")].copy()
        if not tdf.empty:
            tdf["week"] = pd.to_datetime(tdf["date"]).dt.to_period("W").dt.start_time
            tdf = (tdf.groupby(["week", "theme"]).size().unstack(fill_value=0).cumsum()
                   .stack().reset_index(name="requests"))
            st.altair_chart(alt.Chart(tdf).mark_line(strokeWidth=2.5, interpolate="monotone").encode(
                x=alt.X("week:T", title=None), y=alt.Y("requests:Q", title=None),
                color=alt.Color("theme:N", sort=top, legend=alt.Legend(orient="bottom", title=None, columns=2,
                                                                       labelLimit=260)),
                tooltip=["week:T", "theme", "requests"]).properties(height=330), width="stretch")
        else:
            st.caption("No dates in this dataset.")
    with b:
        st.markdown("##### Who is asking")
        st.caption("Mentions per theme by customer plan.")
        st.altair_chart(alt.Chart(idf).mark_bar(cornerRadiusEnd=3).encode(
            x=alt.X("count():Q", title=None, stack="normalize", axis=alt.Axis(format="%")),
            y=alt.Y("theme:N", sort=names, title=None, axis=alt.Axis(labelLimit=240)),
            color=alt.Color("plan:N", legend=alt.Legend(orient="bottom", title=None)),
            tooltip=["theme", "plan", "count()"]).properties(height=30 * len(names) + 20), width="stretch")

    st.markdown("##### Channel × theme")
    st.caption("Where each request comes from. Themes seen across many channels are broad, real needs.")
    heat = idf.groupby(["theme", "channel"]).size().reset_index(name="requests")
    st.altair_chart(alt.Chart(heat).mark_rect(cornerRadius=4).encode(
        x=alt.X("channel:N", title=None, axis=alt.Axis(labelAngle=0)),
        y=alt.Y("theme:N", sort=names, title=None, axis=alt.Axis(labelLimit=300)),
        color=alt.Color("requests:Q", scale=alt.Scale(scheme="purples"), legend=None),
        tooltip=["theme", "channel", "requests"]).properties(height=30 * len(names) + 20)
        + alt.Chart(heat).mark_text(fontWeight=600).encode(x="channel:N", y=alt.Y("theme:N", sort=names),
                                                            text="requests:Q"), width="stretch")

    st.markdown("##### Top accounts asking")
    acc = (idf.groupby("customer").agg(ARR=("arr", "max"), Requests=("id", "count"), Plan=("plan", "first"),
                                       Themes=("theme", lambda s: ", ".join(sorted(set(s)))))
           .sort_values(["ARR", "Requests"], ascending=False).reset_index().rename(columns={"customer": "Account"}))
    st.dataframe(acc.head(15), width="stretch", hide_index=True,
                 column_config={"ARR": st.column_config.NumberColumn(format="$%d"),
                                "Themes": st.column_config.TextColumn(width="large")})

# ---------------------------------------------------------------- evidence + PRD
with tabs[4]:
    pick = st.selectbox("Theme", ranked, format_func=lambda x: f"#{x['rank']}  {x['name']}  ·  {x['tier']}  ·  {x['score']:.0f}")
    a, b = st.columns([3, 2], gap="large")
    with a:
        st.markdown(f"### {pick['name']}")
        st.caption(f"{nice(pick['category'])} · {pick['description']}")
        c = st.columns(4)
        kpi(c[0], "Score", f"{pick['score']:.0f}", pick["tier"])
        kpi(c[1], "Requests", pick["requests"], f"{pick['urgent']} urgent")
        kpi(c[2], "Accounts", pick["accounts"], money(pick["arr"]) + " ARR")
        kpi(c[3], "Effort", pick["effort"], EFFORT_LABEL.get(pick["effort"], ""))
        if pick.get("rationale"):
            st.write("")
            html(f"<div class='summary'><b>Why it ranks here:</b> {escape(pick['rationale'])}</div>")
        st.markdown("##### All feedback in this theme")
        cl = view.clusters[pick["id"]]
        st.dataframe(pd.DataFrame([{"⚡": "⚡" if f.id in cl.urgent_ids else "", "ID": f.id, "Date": f.date,
                                    "Channel": nice(f.channel), "Customer": f.customer, "Feedback": f.text}
                                   for f in (view.feedback[i] for i in cl.feedback_ids)]),
                     width="stretch", hide_index=True,
                     column_config={"Feedback": st.column_config.TextColumn(width="large")})
    with b:
        st.markdown("##### Draft a PRD")
        st.caption(f"{ai} writes a one-page PRD from this theme's customer evidence.")
        prds = ss.setdefault("prds", {})
        if st.button(f"✍️ Draft PRD with {ai}", width="stretch", disabled=not api_key,
                     help=None if api_key else "Add an API key in the sidebar"):
            from roadmap_agent.assistant import draft_prd
            try:
                with st.container(border=True):
                    prds[pick["id"]] = st.write_stream(draft_prd(provider, api_key, view, pick))
            except Exception as e:
                st.error(f"PRD failed: {e}")
        elif pick["id"] in prds:
            with st.container(border=True):
                st.markdown(prds[pick["id"]])
        if pick["id"] in prds:
            st.download_button("Download PRD (.md)", prds[pick["id"]],
                               f"PRD-{pick['name'][:40].replace(' ', '-').replace('/', '-')}.md", width="stretch")
        if not api_key:
            st.info(f"Add a {ai} API key in the sidebar to draft PRDs.", icon="🔑")

# ---------------------------------------------------------------- ask the agent
with tabs[5]:
    st.caption(f"Ask follow-up questions. {ai} answers from this analysis only, citing feedback ids.")
    if not api_key:
        st.info(f"Add a {ai} API key in the sidebar to chat with the agent.", icon="🔑")
    chat = ss.setdefault("chat", [])
    cols = st.columns(len(SUGGESTIONS))
    clicked = next((q for col, q in zip(cols, SUGGESTIONS)
                    if col.button(q, width="stretch", disabled=not api_key)), None)
    box = st.container(height=480, border=True)
    with box:
        if not chat:
            st.caption("No questions yet. Try a suggestion above.")
        for m in chat:
            with st.chat_message(m["role"], avatar="🧭" if m["role"] == "assistant" else None):
                st.markdown(m["content"])
    question = st.chat_input("Ask about the roadmap…", disabled=not api_key) or clicked
    if question:
        chat.append({"role": "user", "content": question})
        from roadmap_agent.assistant import ask
        with box:
            with st.chat_message("user"):
                st.markdown(question)
            with st.chat_message("assistant", avatar="🧭"):
                try:
                    answer = st.write_stream(ask(provider, api_key, view, ranked, chat))
                    chat.append({"role": "assistant", "content": answer})
                except Exception as e:
                    chat.pop()
                    st.error(f"Couldn't get an answer: {e}")
    if chat and st.button("Clear conversation"):
        ss.chat = []
        st.rerun()

# ---------------------------------------------------------------- export
with tabs[6]:
    st.caption("Exports reflect the current weights and filters.")
    clustered = pd.DataFrame([{"id": f.id, "date": f.date, "channel": f.channel, "customer": f.customer,
                               "plan": f.plan, "arr": f.arr, "text": f.text,
                               "cluster": view.clusters[view.assignment[f.id]].name if f.id in view.assignment else "",
                               "category": view.clusters[view.assignment[f.id]].category if f.id in view.assignment else "",
                               "urgent": f.id in view.clusters[view.assignment[f.id]].urgent_ids if f.id in view.assignment else False}
                              for f in view.feedback.values()])
    c = st.columns(4)
    c[0].download_button("📄 Roadmap (Markdown)", report.to_markdown(rep), "roadmap.md", "text/markdown", width="stretch")
    c[1].download_button("🌐 Dashboard (HTML)", report.to_html(rep), "roadmap.html", "text/html", width="stretch")
    c[2].download_button("🧾 Roadmap (JSON)", json.dumps(rep, indent=2), "roadmap.json", "application/json", width="stretch")
    c[3].download_button("📊 Clustered feedback (CSV)", clustered.to_csv(index=False), "clustered_feedback.csv",
                         "text/csv", width="stretch")
    st.markdown("##### Preview")
    st.markdown(report.to_markdown(rep))

In [ ]:
%%writefile requirements.txt
anthropic>=1.13.0
google-genai>=1.30
streamlit>=1.50
pandas
altair

**Sample data** – 80 requests from app store, support tickets, sales calls, NPS, Twitter and forums. Replace with your own or upload a file in the app.

In [ ]:
%%writefile data/feedback.csv
id,date,channel,customer,plan,arr,text
F001,2026-07-02,app_store,anon_1123,free,0,Please add a dark mode! My eyes hurt using this at night.
F002,2026-07-03,support_ticket,Acme Corp,enterprise,120000,"We need SAML SSO with Okta before our security team will approve rollout to 2,000 seats."
F003,2026-07-03,sales_call,Northwind Traders,enterprise,85000,Procurement is blocked on single sign-on. They use Azure AD. Deal at risk without it.
F004,2026-07-04,community_forum,jdoe_pm,pro,1200,Would love a Slack integration so task updates post to our team channel.
F005,2026-07-05,nps_survey,Brightside Labs,pro,4800,Exporting reports to Excel is painful - copy/paste every week. Give us a CSV export.
F006,2026-07-05,app_store,anon_2201,free,0,The app is unusable on the subway. Need offline mode on mobile!
F007,2026-07-06,support_ticket,Globex,enterprise,210000,Our execs want custom dashboards with the KPIs they care about instead of the fixed overview page.
F008,2026-07-07,twitter,@sarah_builds,pro,600,Dark theme when?? Every other tool I use has one.
F009,2026-07-08,support_ticket,Initech,business,24000,Can you add webhooks so we can trigger our internal workflows when a ticket closes?
F010,2026-07-08,nps_survey,Hooli,enterprise,150000,"Love the product, but no SSO is a dealbreaker for renewal. Okta please."
F011,2026-07-09,community_forum,maria_ops,business,9000,Need recurring tasks - we create the same weekly checklist by hand every Monday.
F012,2026-07-10,support_ticket,Umbrella Inc,business,18000,Dashboard takes 12 seconds to load with our 40k records. Very slow.
F013,2026-07-10,app_store,anon_3302,free,0,Would pay for Pro if there was a dark mode.
F014,2026-07-11,sales_call,Stark Industries,enterprise,300000,Security questionnaire requires enforced 2FA for all users and SSO. Both are must-haves.
F015,2026-07-12,community_forum,devon_k,pro,1200,Microsoft Teams integration please. We are not a Slack shop.
F016,2026-07-12,nps_survey,Vandelay Imports,business,12000,Please let us export everything to PDF for our board packs.
F017,2026-07-13,support_ticket,Wayne Enterprises,enterprise,260000,Need a public REST API to sync data with our data warehouse.
F018,2026-07-14,app_store,anon_4410,free,0,App crashes every time I open the calendar view on Android 15.
F019,2026-07-14,twitter,@techlead_amy,pro,600,"Offline support for the iOS app would be huge, I travel a lot."
F020,2026-07-15,support_ticket,Soylent Co,business,15000,Can we build our own reports? The built-in ones don't match how we measure throughput.
F021,2026-07-15,community_forum,raj_m,pro,1200,Templates for recurring projects would save us hours each sprint.
F022,2026-07-16,nps_survey,Massive Dynamic,enterprise,180000,Add two-factor authentication. Our auditors flagged the lack of MFA.
F023,2026-07-17,sales_call,Cyberdyne,enterprise,95000,Prospect requires SCIM provisioning and SAML SSO to move forward.
F024,2026-07-17,app_store,anon_5120,free,0,Dark mode is the only thing missing. 4 stars until then.
F025,2026-07-18,support_ticket,Acme Corp,enterprise,120000,Following up on SSO request - our CISO is asking for a timeline.
F026,2026-07-19,community_forum,lin_design,pro,1200,Bulk export of all project data as CSV would make migrations and backups so much easier.
F027,2026-07-19,twitter,@nightowl_dev,free,0,Begging for a dark UI. Please.
F028,2026-07-20,support_ticket,Pied Piper,business,20000,Slack notifications when someone mentions me would be amazing.
F029,2026-07-21,nps_survey,Brightside Labs,pro,4800,Interface feels sluggish when switching between projects.
F030,2026-07-21,sales_call,Oceanic Airlines,enterprise,140000,They need the product in Spanish and Portuguese for LATAM teams.
F031,2026-07-22,app_store,anon_6001,free,0,Great app! Really helped our small team get organized.
F032,2026-07-23,support_ticket,Globex,enterprise,210000,Need to schedule dashboards to email to leadership every Friday.
F033,2026-07-23,community_forum,ken_w,business,9000,Please support webhooks + an API key per workspace so we can automate.
F034,2026-07-24,support_ticket,Initech,business,24000,Our API integration keeps getting rate limited at 60 req/min. Please raise limits.
F035,2026-07-25,twitter,@pm_jules,pro,600,Need a way to export my data. Feels locked in right now.
F036,2026-07-25,app_store,anon_7123,free,0,No offline mode = I can't use it on flights. Deleting.
F037,2026-07-26,nps_survey,Hooli,enterprise,150000,Two-factor login should be enforceable by admins.
F038,2026-07-27,support_ticket,Umbrella Inc,business,18000,Search is very slow - takes several seconds to return results.
F039,2026-07-27,community_forum,ana_lead,business,9000,Recurring tasks are a basic feature - surprised it's missing.
F040,2026-07-28,sales_call,Tyrell Corp,enterprise,110000,"Lost the evaluation to a competitor mainly because of no Slack/Teams integration."
F041,2026-07-29,support_ticket,Vandelay Imports,business,12000,I was charged twice this month. Please refund.
F042,2026-07-29,app_store,anon_8210,free,0,Night mode please - the white background is blinding.
F043,2026-07-30,nps_survey,Soylent Co,business,15000,Custom charts and the ability to pick which metrics show on the home dashboard.
F044,2026-07-31,community_forum,tom_r,pro,1200,Is there a way to use the app offline on Android? Would really help field teams.
F045,2026-08-01,support_ticket,Wayne Enterprises,enterprise,260000,Still waiting on SSO via Okta. Blocking expansion to 3 more business units.
F046,2026-08-02,twitter,@ux_kate,pro,600,Please localize to German - half our team would prefer it.
F047,2026-08-03,sales_call,Monsters Inc,business,30000,Wants to export Gantt charts to PDF for client presentations.
F048,2026-08-04,app_store,anon_9001,free,0,"Crashes when uploading attachments larger than 10MB on iPhone."
F049,2026-08-04,support_ticket,Pied Piper,business,20000,We'd love a native Microsoft Teams app with notifications.
F050,2026-08-05,nps_survey,Massive Dynamic,enterprise,180000,Admins need to enforce SSO-only login for compliance.
F051,2026-08-06,community_forum,priya_s,pro,1200,Dark mode + high-contrast theme for accessibility please.
F052,2026-08-07,support_ticket,Stark Industries,enterprise,300000,Need SCIM user provisioning tied to our IdP.
F053,2026-08-08,twitter,@remote_ray,pro,600,Recurring reminders would make this my default task app.
F054,2026-08-08,app_store,anon_1010,free,0,"Loading takes forever on 4G, can it be faster?"
F055,2026-08-09,nps_survey,Initech,business,24000,The API is missing endpoints for comments and attachments.
F056,2026-08-10,sales_call,Gringotts,enterprise,220000,Bank requires mandatory MFA and audit logs before purchase.
F057,2026-08-11,support_ticket,Globex,enterprise,210000,Custom report builder with filters and grouping is our #1 ask.
F058,2026-08-12,community_forum,omar_h,business,9000,An Excel export of time tracking data please.
F059,2026-08-13,app_store,anon_1111,free,0,Offline access to my notes would be a game changer.
F060,2026-08-13,twitter,@founder_lee,pro,600,Your onboarding is the best I've seen. Keep it up!
F061,2026-08-14,support_ticket,Oceanic Airlines,enterprise,140000,Can the UI be translated to French for our Montreal office?
F062,2026-08-15,nps_survey,Monsters Inc,business,30000,Integration with Slack would cut our status meetings in half.
F063,2026-08-16,community_forum,zoe_t,pro,1200,Please add a dark mode option on desktop too.
F064,2026-08-17,support_ticket,Hooli,enterprise,150000,Audit log of admin actions is required by our compliance team.
F065,2026-08-18,sales_call,Acme Corp,enterprise,120000,Expansion of 1500 seats contingent on SSO shipping this quarter.
F066,2026-08-19,app_store,anon_1212,free,0,Repeat tasks feature please - daily habits would be perfect here.
F067,2026-08-20,support_ticket,Cyberdyne,enterprise,95000,Need audit logs exportable to our SIEM.
F068,2026-08-21,twitter,@dataguy,pro,600,"API docs are thin and there's no webhook support, so building on you is hard."
F069,2026-08-22,nps_survey,Umbrella Inc,business,18000,Performance with large projects is getting worse every month.
F070,2026-08-23,community_forum,bea_c,pro,1200,How do I change my billing email?
F071,2026-08-24,support_ticket,Soylent Co,business,15000,Please let us save custom report views and share them with the team.
F072,2026-08-25,app_store,anon_1313,free,0,Pleeease dark mode
F073,2026-08-26,sales_call,Wonka Industries,business,40000,Requires Google Workspace SSO for their 300 users.
F074,2026-08-27,support_ticket,Vandelay Imports,business,12000,Export to CSV is missing custom fields.
F075,2026-08-28,community_forum,eli_m,business,9000,Teams integration so approvals can happen in chat.
F076,2026-08-29,nps_survey,Gringotts,enterprise,220000,Two-factor authentication via authenticator app is required.
F077,2026-08-30,app_store,anon_1414,free,0,Offline mode for when I'm on job sites without signal.
F078,2026-08-31,support_ticket,Tyrell Corp,enterprise,110000,Spanish localization needed for our Madrid team.
F079,2026-09-01,twitter,@mobile_max,pro,600,App freezes when scrolling long lists on Android.
F080,2026-09-02,support_ticket,Wayne Enterprises,enterprise,260000,Higher API rate limits and bulk endpoints for our nightly sync.

## 3 · Load API keys from Colab secrets
Add `ANTHROPIC_API_KEY` (Claude) and/or `GEMINI_API_KEY` (Gemini, free at aistudio.google.com) in the 🔑 Secrets panel.

In [ ]:
import os
from google.colab import userdata
for name in ["ANTHROPIC_API_KEY", "GEMINI_API_KEY"]:
    try:
        os.environ[name] = userdata.get(name)
        print(f"✅ {name} loaded")
    except Exception:
        print(f"–  {name} not set (that provider can still use a key pasted in the app sidebar)")
PROVIDER = "gemini" if os.environ.get("GEMINI_API_KEY") and not os.environ.get("ANTHROPIC_API_KEY") else "claude"
print("Notebook agent run will use:", PROVIDER)

## 4 · Try it from the notebook
Offline baseline first (instant, free), then the AI agent.

In [ ]:
!python main.py data/feedback.csv --offline --out output/offline

In [ ]:
# Runs the full AI agent (takes a few minutes; watch its tool calls stream by).
if os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("GEMINI_API_KEY"):
    !python main.py data/feedback.csv --provider {PROVIDER} --out output/agent
else:
    print("Skipped – no API key.")

In [ ]:
from IPython.display import HTML, display
from pathlib import Path
path = Path("output/agent/roadmap.html")
if not path.exists():
    path = Path("output/offline/roadmap.html")
display(HTML(path.read_text()))

## 5 · Launch the Streamlit front-end
Starts Streamlit in the background and opens a free public **Cloudflare tunnel** to it.
Open the `https://….trycloudflare.com` link that is printed. The link stays alive while this cell's runtime is running.

In [ ]:
import re, subprocess, threading, time

!test -f cloudflared || (wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared)

subprocess.run("pkill -f 'streamlit run' ; pkill -f cloudflared", shell=True)
streamlit = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
    stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(4)

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
# Keep draining the tunnel's log so it never blocks.
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()
print(f"🚀 Open the app: {url}" if url else "Tunnel failed – see the output above.")

In [ ]:
# Troubleshooting: show the Streamlit server log
!tail -n 30 streamlit.log